# Module 01 — How Spark Works

Focus: distributed execution, reading the Spark UI, and how configs change the physical plan.

Open http://localhost:4040 before running any **action**.

In [1]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')

from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config

spark = begin_module('module-01-how-spark-works')
ui_links()  # App UI at :4040 is live after this cell

KeyError: 'kernel_shutdown_requested'

## Load Data

NYC yellow taxi trips (public TLC dataset). On first run, files are downloaded to `data/cache/` (~100 MB total). Spark cannot read Parquet directly from HTTPS URLs.

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
print(f'Trips loaded — run an action to see row count')

In [ ]:
print(f'Trips: {trips.count():,} rows')
# First action → first job appears in the UI

## Spark UI: Jobs, Stages, Tasks

| UI level | Meaning |
|----------|--------|
| **Job** | Created by an **action** (count, show, write) |
| **Stage** | Pipeline of work separated by a **shuffle** (data moving across the cluster) |
| **Task** | One unit of work on one partition, on one executor |

Transformations alone do not create jobs — they only build the plan.

## 1. Aggregation

A `groupBy` collects rows across the cluster. Check **Jobs** and **Stages** tabs.

In [ ]:
vendor_counts = trips.groupBy('VendorID').count().orderBy('count', ascending=False)
vendor_counts.show()
# How many jobs? stages? tasks per stage?

In [ ]:
explain_plan(vendor_counts)

## 2. Window Function

Window functions need rows from the same group on the same machine. That data movement creates a new **stage boundary** in the UI — look for **Shuffle Read/Write** metrics.

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

w = Window.partitionBy('VendorID').orderBy(col('tpep_pickup_datetime').desc())
ranked = trips.withColumn('rn', row_number().over(w)).filter('rn <= 3')
ranked.select('VendorID', 'tpep_pickup_datetime', 'rn').show(10)

In [ ]:
explain_plan(ranked)

## 3. Combining Two Datasets

When Spark combines two datasets, data must be aligned across executors. Don't worry about *how* yet — just observe that new stages appear and shuffle bytes increase.

In [ ]:
zones = spark.read.option('header', True).csv(config.zones_path())
enriched = trips.join(
    zones.withColumnRenamed('LocationID', 'PULocationID'),
    on='PULocationID',
    how='left',
)
enriched.select('PULocationID', 'Zone', 'Borough', 'total_amount').show(5)

In [ ]:
explain_plan(enriched)

## 4. The Physical Plan Is Not Fixed

The same logical query can produce different physical plans depending on **configuration**. Spark chooses at runtime — your code alone does not determine the plan.

In [ ]:
# Config A: fewer shuffle partitions
spark.conf.set('spark.sql.shuffle.partitions', '8')
plan_a = trips.groupBy('PULocationID').count()
plan_a.explain('formatted')

In [ ]:
# Config B: more shuffle partitions
spark.conf.set('spark.sql.shuffle.partitions', '200')
plan_b = trips.groupBy('PULocationID').count()
plan_b.explain('formatted')
# Compare partition counts in the plan

In [ ]:
# Run both and compare task counts in the Stages tab
plan_a.count()
plan_b.count()

In [ ]:
# Reset to default
spark.conf.unset('spark.sql.shuffle.partitions')

## 5. Live Discussion (Walk Through Together)

Pick one query above and answer together:
1. How many jobs did the action create?
2. Which stage has shuffle read/write?
3. How many tasks ran in each stage?
4. What changed when we altered `spark.sql.shuffle.partitions`?

## Key Takeaways

1. Transformations are lazy; **actions** trigger jobs.
2. **Stages** split at shuffle boundaries — data moving across the cluster is expensive.
3. The physical plan depends on **configs and runtime conditions**, not just your code.